# SGD and Regression on Diabetes database
The structure of this notebook reflects the implementation of SGD and linear regression from scratch in Python as of ```Friday, March 13, 2026``` in this repository.
## Gradient Descent
1. Develop Numerical Differentiation.
2. Extend to multivariable case for calculating gradients
3. Define an iterative algorithm for gradient descent.

In [1]:
import numpy as np
import random as rnd

### Numerical differentiation
- Using standard LHL-RHL definition of limits, then averaging for better stability. Can do with multiple sample points like $x_0-2h,x_0-h,x_0,x_0+h,x_0+2h$, but not strictly necessary for the moment.
- This assumes that the function we provide will be differentiable at the point in question.

In [2]:
def numerical_derivative(f, x, axis, diff_step = 1e-5):
    if axis >= x.shape[0] :
        raise IndexError(f"Can't differentiate wrt x_{axis+1} when x has shape {x.shape}")
    else :
        basis_vector = np.zeros_like(x)
        basis_vector[axis] = diff_step

        left = (f(x) - f(x - basis_vector))/diff_step

        right = (f(x + basis_vector) - f(x))/diff_step

        return 0.5 * (left + right)

#-------------------------------------------------------------------------------------------------

### Gradient
- Simply performing numerical differentiation along different axes/coordinates.

In [3]:
# Takes a function and a point
# Returns an array (same shape as input vector) containing partial derivatives evaluated at the point
def grad(f, x) :
    gg = np.zeros(x.shape)
    # print(f"x has length {gg.shape[0]}")
    for i in range(x.shape[0]) :
        gg[i] = numerical_derivative(f, x, i)
        # print(f"using partial deriv {numerical_derivative(f, x, i)}")
    return gg

#-------------------------------------------------------------------------------------------------


### Gradient Descent
- Given a (differentiable) function $f$, an initial point $x_0$, one can compute the gradient at the point, find the direction of greatest change of $x_0$ (in fact it is given precisely by the gradient), and "ascend" or "descend" in that direction.
- We stop when a certain number of iterations have been performed, quit early if gradient is small. Better stopping conditions are possible:
- example : stop when gradient is small enough (what if it takes too long?)
- example : stop when the objective value is cycling/looping/not reducing fast enough over the last $k$-iterations, for some set $k$.

In [4]:
# Takes a function, initial point, number of iterations and step size as input
# Performs gradient descent, returns an estimate of a local arg_min(f), given the starting point.
def gd(f, init_x, step_size : float, max_iters : int = 200) :
    current_vector = init_x
    iteration = 0
    gradient = np.ones(init_x.shape)
    while iteration < max_iters and np.sum(gradient**2) > 1e-4:
        gradient = grad(f, current_vector)
        # gradient /= np.sum(gradient**2)
        current_vector = current_vector - step_size * gradient
        iteration += 1
    return current_vector

#-------------------------------------------------------------------------------------------------

## Stochastic Gradient Descent
Given a function that depends on some data (separate from the optimizing parameter), we can either optimize the whole function with the complete data. Alternatively, we use sub-samples to define a restricted version of the whole function, and optimize it using gradient descent. In particular, we choose a batch size and the number of epochs (number of times we will "learn" from the same textbook). Then :
- In each epoch, partition the complete sample into random batches of predetermined size, avoiding repetition inside the batches.
- For each batch : Make an error function (in our case, but the theory holds generally) using just the batch data.
- Minimize the batch error function for 1 (or more) iteration(s) using gradient descent.
- New batch : repeat.
- New epoch : repeat.

In [5]:
def randomizer (num_of_samples : int, batch_size : int) : # Use each epoch
    subsample = rnd.sample(range(num_of_samples), batch_size)
    return subsample

In [6]:
def ols_error_function(X,y) : # Returns a function that will take "Data", that is [X,y]
    def loss (theta) :
        return np.sum((y - X @ theta)**2) # This X will have to have an extra column of 1s.
    return loss

In [7]:
def sgd(XX, y, n_epochs : int, batch_size : int, learning_rate : float) :
    # First prepare for an added constant. Add a column of ones to X.
    X = np.concatenate((np.ones((XX.shape[0], 1), dtype=float), XX), axis=1)

    epoch = 0
    n_samples = X.shape[0]
    current_theta = np.zeros(X.shape[1])
    # Number of iterations to be used for each use of gd.
    # print("starting sgd. epoch 1, current theta is 0")
    while epoch < n_epochs :
        indices = randomizer(n_samples , batch_size)

        X_train = X[indices]
        y_train = y[indices]

        error = ols_error_function(X_train,y_train)
        # now do gradient descent :
        # print(f"epoch {epoch}, current theta is {str(current_theta)}")
        # print()

        current_theta = gd(error, current_theta, learning_rate)

        epoch = epoch + 1
    # print()
    # print(f"loop finished. theta is {str(current_theta)}")
    return current_theta # This is t_0, t_1, t_2, ..., t_n

In [8]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split


diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target

print("Loaded the diabetes database from sklearn.datasets")
print()
# 2. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Data has been split. Training on {X_train.shape[0]} samples and testing on {X_test.shape[0]} samples")
print()
print("Now beginning SGD...")

theta = sgd(X_train, y_train, 50, 150, 0.001)
print("SGD complete; will show a comparison of true-vs-predicted values now :")
print()


n_samples_to_show = 10
shuffled_indices = np.random.permutation(n_samples_to_show)
for i in shuffled_indices:
    print(y_test[i], " ----------- ", theta[0] + X_test[i]@theta[1:])


Loaded the diabetes database from sklearn.datasets

Data has been split. Training on 353 samples and testing on 89 samples

Now beginning SGD...
SGD complete; will show a comparison of true-vs-predicted values now :

94.0  -----------  80.30862013926514
219.0  -----------  131.66106767371232
111.0  -----------  117.84093812922562
96.0  -----------  108.56954414420025
84.0  -----------  88.59970460509834
230.0  -----------  285.2320628850206
70.0  -----------  179.10146938556812
242.0  -----------  250.43549275340467
272.0  -----------  180.24300145353305
202.0  -----------  132.6937771573085


### Thoughts :
Throughout the minimization process we have had faith that a minimum exists, and will be achieved. For convex loss functions such as in regression, kernel methods, etc, we know this, and SGD (with appropriate step size) does find at least a global minimum (if it exists in $\mathbb{R}$), so we are safe.
Also :
> What might be a good estimate of how many iterations of GD should be used? It has been manually set as 200, but is that optimal?